<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# NLP Basics

**Prediction of Text (based on Characters)**

&copy; Dr. Yves J. Hilpisch

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>

## Imports

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.eras.preprocessing.sequence import TimeseriesGenerator
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [ ]:
np.set_printoptions(suppress=True)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
from pylab import plt
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = 'svg'

## The Text

In [ ]:
# text = 'this is a short sentence. this is another one. and yet another one.'

In [ ]:
# text = '''this is a short sentence. this is another one. and yet another one. but what about adding even more text to the string? this might be more difficult.'''

In [ ]:
text = '''import math
from scipy.stats import norm

def black_scholes_call(S, K, T, r, sigma):
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    call_price = S * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
    return call_price'''

In [ ]:
text = text.lower()  # .replace('\n', ' ')

In [ ]:
text

In [ ]:
length = 10

In [ ]:
snippets = list()
next_chars = list()

In [ ]:
for i in range(len(text) - length):
    snippets.append(text[i:i + length])
    next_chars.append(text[i + length])

In [ ]:
snippets[:5]

In [ ]:
next_chars[:5]

In [ ]:
chars = sorted(set(text))
chars[:10]

In [ ]:
len(chars)

In [ ]:
cti = {c: i for i, c in enumerate(chars)}

In [ ]:
itc = {i: c for i, c in enumerate(chars)}

In [ ]:
X = list()
for s in snippets:
    il = list()
    for c in s:
        il.append(cti[c])
    X.append(il)
X = np.array(X)

In [ ]:
X[:5]

In [ ]:
y = np.array([cti[c] for c in next_chars])

In [ ]:
y[:5]

## RNNs for Classification

In [ ]:
encoder = OneHotEncoder(sparse_output=False)

In [ ]:
y_ = encoder.fit_transform(y.reshape(-1, 1))

In [ ]:
len(chars)

In [ ]:
y_.shape

In [ ]:
model = Sequential()
model.add(LSTM(64, activation='relu',
               return_sequences=True, input_shape=(length, 1)))
model.add(LSTM(64, activation='relu'))
model.add(Dense(len(chars), activation='softmax'))
model.compile(loss='categorical_crossentropy',
              optimizer=Adam(learning_rate=0.0001))

In [ ]:
%time model.fit(X, y_, epochs=750, verbose=False)

In [ ]:
model.predict(X)[:1]

In [ ]:
p = np.argmax(model.predict(X), axis=1)
p[:10]

In [ ]:
tp = [itc[max(i, 0)] for i in p]
textp = ''.join(tp)
textp

In [ ]:
print(textp)

In [ ]:
sum([text[length:][i] == textp[i] for i in range(len(textp))]) / len(textp)

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>